# Mini-lab — Do Teste 1 ao mundo real

> Resolva os dois modelos da NeuralCloud (bruto × líquido) e compare.

Este notebook acompanha o roteiro **Mini-lab · Custos invisíveis**. O fluxo é:

1. Instalar o AMPL e o solver.
2. Escrever os quatro arquivos AMPL (`bruto.mod`, `bruto.dat`, `liquido.mod`, `liquido.dat`) usando o comando mágico `%%writefile`.
3. Resolver os dois modelos.
4. Comparar Z* e a alocação ótima.

Antes de rodar, leia o roteiro em PDF — ele explica de onde saem os parâmetros e o que você está procurando.


## 1. Instalação

A célula abaixo instala o `amplpy`, baixa o solver HiGHS e instancia o objeto `ampl` já configurado. Rode uma vez no início da sessão.


In [ ]:
!pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=["highs"],
    license_uuid="default",
)


## 2. Modelo do teste — margem bruta

A função objetivo aqui é a do enunciado original do Teste 1: maximizar a soma das margens unitárias (80, 120, 200 em `$/inst`) vezes as alocações.

### `bruto.mod`

O comando `%%writefile` no topo da célula faz o Colab gravar **todo o conteúdo da célula** em um arquivo no servidor. Não é Python — é uma instrução do ambiente.


In [ ]:
%%writefile bruto.mod
set DC;
set PLANO;

param margem   {PLANO} >= 0;
param kw       {PLANO} >= 0;
param dem_max  {PLANO} >= 0;
param cap_inst {DC}    >= 0;
param cap_kw   {DC}    >= 0;
param tol_balanc       >= 0;
param frac_max         >= 0, <= 1;

var x {DC, PLANO} >= 0;
var y {i in DC} = sum {j in PLANO} x[i,j];
var carga {j in PLANO} = sum {i in DC} x[i,j];

maximize Margem_Bruta:
    sum {i in DC, j in PLANO} margem[j] * x[i,j];

s.t. Capac    {i in DC}: y[i] <= cap_inst[i];
s.t. Potencia {i in DC}: sum {j in PLANO} kw[j]*x[i,j] <= cap_kw[i];
s.t. Demanda  {j in PLANO}: carga[j] <= dem_max[j];
s.t. BalSup: y["DC1"] - y["DC3"] <= tol_balanc;
s.t. BalInf: y["DC3"] - y["DC1"] <= tol_balanc;
s.t. TolFalhas {i in DC, j in PLANO}: x[i,j] <= frac_max * carga[j];


### `bruto.dat`


In [ ]:
%%writefile bruto.dat
set DC := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param tol_balanc := 100 ;
param frac_max   := 0.60 ;

param :       margem  kw  dem_max :=
  Basic         80     5     700
  Pro          120     8     900
  Ultra        200    12     600 ;

param :  cap_inst  cap_kw :=
  DC1      600      4500
  DC2      800      5000
  DC3      500      3000 ;


### Resolver o modelo bruto

Lemos os dois arquivos, resolvemos, e mostramos o lucro e a alocação.


In [ ]:
import pandas as pd

ampl.reset()
ampl.read("bruto.mod")
ampl.read_data("bruto.dat")
ampl.set_option("solver", "highs")
ampl.solve()

Z_bruto = ampl.get_objective("Margem_Bruta").value()
print(f"Z* (margem bruta) = $ {Z_bruto:,.2f} / semana")

aloc_bruto = ampl.get_variable("x").get_values().to_pandas().unstack()
aloc_bruto.columns = aloc_bruto.columns.droplevel()
print("\nAlocação ótima (instâncias por DC × plano):")
display(aloc_bruto.round(1))


## 3. Modelo realista — margem líquida

Agora trocamos a função objetivo para descontar energia, água, depreciação de GPU e OPEX. As restrições são as mesmas.

### `liquido.mod`


In [ ]:
%%writefile liquido.mod
set DC;
set PLANO;

param preco      {PLANO} >= 0;
param kw         {PLANO} >= 0;
param dem_max    {PLANO} >= 0;
param dep_gpu    {PLANO} >= 0;
param custo_op   {PLANO} >= 0;
param cap_inst   {DC}    >= 0;
param cap_kw     {DC}    >= 0;
param preco_kwh  {DC}    >= 0;
param wue        {DC}    >= 0;
param preco_agua {DC}    >= 0;
param H                  >= 0;
param tol_balanc         >= 0;
param frac_max           >= 0, <= 1;

var x {DC, PLANO} >= 0;
var y {i in DC} = sum {j in PLANO} x[i,j];
var carga {j in PLANO} = sum {i in DC} x[i,j];
var kwh {i in DC} = H * sum {j in PLANO} kw[j]*x[i,j];

maximize Margem_Liquida:
    sum {i in DC, j in PLANO} preco[j] * x[i,j]
  - sum {i in DC} preco_kwh[i] * kwh[i]
  - sum {i in DC} preco_agua[i] * wue[i] * kwh[i] / 1000
  - sum {i in DC, j in PLANO} dep_gpu[j] * x[i,j]
  - sum {i in DC, j in PLANO} custo_op[j] * x[i,j];

s.t. Capac    {i in DC}: y[i] <= cap_inst[i];
s.t. Potencia {i in DC}: sum {j in PLANO} kw[j]*x[i,j] <= cap_kw[i];
s.t. Demanda  {j in PLANO}: carga[j] <= dem_max[j];
s.t. BalSup: y["DC1"] - y["DC3"] <= tol_balanc;
s.t. BalInf: y["DC3"] - y["DC1"] <= tol_balanc;
s.t. TolFalhas {i in DC, j in PLANO}: x[i,j] <= frac_max * carga[j];


### `liquido.dat`


In [ ]:
%%writefile liquido.dat
set DC := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param H          := 168 ;
param tol_balanc := 100 ;
param frac_max   := 0.60 ;

param :  preco  kw  dem_max  dep_gpu  custo_op :=
  Basic    90    5    700      10        5
  Pro     180    8    900      30       10
  Ultra   360   12    600      80       20 ;

param :  cap_inst  cap_kw  preco_kwh  wue  preco_agua :=
  DC1      600     4500     0.12     1.8    3.00
  DC2      800     5000     0.08     1.5    2.50
  DC3      500     3000     0.06     0.5    1.80 ;


### Resolver o modelo realista


In [ ]:
ampl.reset()
ampl.read("liquido.mod")
ampl.read_data("liquido.dat")
ampl.set_option("solver", "highs")
ampl.solve()

Z_liquido = ampl.get_objective("Margem_Liquida").value()
print(f"Z* (margem líquida) = $ {Z_liquido:,.2f} / semana")

aloc_liquido = ampl.get_variable("x").get_values().to_pandas().unstack()
aloc_liquido.columns = aloc_liquido.columns.droplevel()
print("\nAlocação ótima (instâncias por DC × plano):")
display(aloc_liquido.round(1))


## 4. Compare

Monte a tabela comparativa pedida no roteiro: Z* dos dois modelos lado a lado, e o total alocado de cada plano. A célula abaixo é um esqueleto — você só precisa rodá-la, depois que as duas anteriores tiverem rodado.


In [ ]:
totais_bruto    = aloc_bruto.sum(axis=0)
totais_liquido  = aloc_liquido.sum(axis=0)

comparacao = pd.DataFrame({
    "Modelo do teste (bruto)":   [Z_bruto, *totais_bruto.values],
    "Modelo realista (líquido)": [Z_liquido, *totais_liquido.values],
}, index=["Z* ($/semana)", "Total Basic", "Total Pro", "Total Ultra"])

display(comparacao.round(2))

print(f"\nDiferença Z*bruto - Z*líquido = $ {Z_bruto - Z_liquido:,.2f}")
print(f"Razão Z*bruto / Z*líquido     = {Z_bruto/Z_liquido:.2f}x")


## 5. Reflexão

Responda em **3–5 linhas** abaixo (clique duas vezes nesta célula para editar):

**a)** O Z* do modelo realista é cerca de 1/3 do Z* bruto. Onde foi parar a diferença?

> *(sua resposta)*

**b)** O plano Basic vai a zero no modelo realista. Olhando para os parâmetros do DC1, calcule a margem líquida unitária de uma instância de Basic em DC1. O que isso explica?

> *(sua resposta)*

**c)** Qual modelo um operador real deveria usar para tomar decisões — e qual é o risco de operar guiado pelo modelo do teste?

> *(sua resposta)*
